# LOREM-SOG vs LOREM-CU：best_model 指标对比（能量/力 RMSE & MAE）

这个 notebook 做两件事：
1. 读取两边 `best_model`（这里按 `run/checkpoints/MAE_F`）的验证指标；
2. 输出能量和力的 RMSE/MAE 对比结论。

> 说明：你当前目录里已有训练产物与 checkpoint 指标文件，所以这里先做“基于 best checkpoint 的可复现对比”。
> 若你想完整“再跑一遍推理/评估流程”，末尾有可选命令模板单元格。

In [2]:
from pathlib import Path
import yaml
import pandas as pd

SOG_DIR = Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2')
CU_DIR = Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp2')

# 这里把 best_model 定义为 MAE_F checkpoint（通常对应“best force model”）
SOG_METRICS = SOG_DIR / 'run/checkpoints/MAE_F/plot/valid/metrics.yaml'
CU_METRICS = CU_DIR / 'run/checkpoints/MAE_F/plot/valid/metrics.yaml'

for p in [SOG_METRICS, CU_METRICS]:
    if not p.exists():
        raise FileNotFoundError(f'Missing metrics file: {p}')

print('SOG metrics:', SOG_METRICS)
print('CU  metrics:', CU_METRICS)

SOG metrics: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2/run/checkpoints/MAE_F/plot/valid/metrics.yaml
CU  metrics: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp2/run/checkpoints/MAE_F/plot/valid/metrics.yaml


In [3]:
def load_metrics(path: Path):
    with path.open('r') as f:
        d = yaml.safe_load(f)
    return {
        'energy_mae': float(d['energy']['mae']),
        'energy_rmse': float(d['energy']['rmse']),
        'force_mae': float(d['forces']['mae']),
        'force_rmse': float(d['forces']['rmse']),
    }

sog = load_metrics(SOG_METRICS)
cu = load_metrics(CU_METRICS)

df = pd.DataFrame([
    {'model': 'LOREM-SOG', **sog},
    {'model': 'LOREM-CU', **cu},
])

df

,model,energy_mae,energy_rmse,force_mae,force_rmse
0,LOREM-SOG,6.919211,41.037463,14.568763,50.998505
1,LOREM-CU,9.661443,42.651845,12.011340,49.378926


In [3]:
cmp = pd.DataFrame({
    'metric': ['energy_mae', 'energy_rmse', 'force_mae', 'force_rmse'],
    'LOREM-SOG': [sog['energy_mae'], sog['energy_rmse'], sog['force_mae'], sog['force_rmse']],
    'LOREM-CU': [cu['energy_mae'], cu['energy_rmse'], cu['force_mae'], cu['force_rmse']],
})
cmp['better'] = cmp.apply(lambda r: 'LOREM-SOG' if r['LOREM-SOG'] < r['LOREM-CU'] else 'LOREM-CU', axis=1)
cmp['delta(SOG-CU)'] = cmp['LOREM-SOG'] - cmp['LOREM-CU']
cmp

,metric,LOREM-SOG,LOREM-CU,better,delta(SOG-CU)
0,energy_mae,6.919211,9.661443,LOREM-SOG,-2.742232
1,energy_rmse,41.037463,42.651845,LOREM-SOG,-1.614382
2,force_mae,14.568763,12.011340,LOREM-CU,2.557423
3,force_rmse,50.998505,49.378926,LOREM-CU,1.619579


In [4]:
print('结论（基于 MAE_F best checkpoint 的 valid 指标）:')
for _, r in cmp.iterrows():
    sign = '更低(更好)' if r['delta(SOG-CU)'] < 0 else '更高(更差)'
    print(f"- {r['metric']}: SOG={r['LOREM-SOG']:.6f}, CU={r['LOREM-CU']:.6f} -> {r['better']} ({sign})")

结论（基于 MAE_F best checkpoint 的 valid 指标）:
- energy_mae: SOG=6.919211, CU=9.661443 -> LOREM-SOG (更低(更好))
- energy_rmse: SOG=41.037463, CU=42.651845 -> LOREM-SOG (更低(更好))
- force_mae: SOG=14.568763, CU=12.011340 -> LOREM-CU (更高(更差))
- force_rmse: SOG=50.998505, CU=49.378926 -> LOREM-CU (更高(更差))


## 重新跑一遍：只加载 best checkpoint 做前向评估（eval-only）

下面开始做真正的 eval-only：
- 不训练；
- 直接加载两边 `run/checkpoints/MAE_F` 下的 `model.msgpack`；
- 在同一个 `cumulene_test.xyz` 上计算 energy/forces 的 RMSE 与 MAE。

In [6]:
import sys
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from ase.io import read

LOREM_ROOT = Path('/data/home/public/qiuqizhi/LOREM')
LOREM_CODE = LOREM_ROOT / 'lorem'
if str(LOREM_CODE) not in sys.path:
    sys.path.insert(0, str(LOREM_CODE))

from calculator import Calculator
from marathon.data import datasets

SOG_CKPT = SOG_DIR / 'run/checkpoints/MAE_F'
CU_CKPT = CU_DIR / 'run/checkpoints/MAE_F'

# 某些环境下 marathon.data.datasets 可能是 None，这里做回退。
if datasets is None:
    TEST_XYZ = LOREM_ROOT / 'datasets' / 'cumulene_test.xyz'
else:
    TEST_XYZ = Path(datasets) / 'cumulene_test.xyz'

for p in [
    SOG_CKPT / 'model/model.msgpack',
    SOG_CKPT / 'model/model.yaml',
    SOG_CKPT / 'model/baseline.yaml',
    CU_CKPT / 'model/model.msgpack',
    CU_CKPT / 'model/model.yaml',
    CU_CKPT / 'model/baseline.yaml',
    TEST_XYZ,
]:
    if not p.exists():
        raise FileNotFoundError(f'Missing: {p}')

print('SOG checkpoint:', SOG_CKPT)
print('CU  checkpoint:', CU_CKPT)
print('Dataset       :', TEST_XYZ)

SOG checkpoint: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2/run/checkpoints/MAE_F
CU  checkpoint: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp2/run/checkpoints/MAE_F
Dataset       : /data/home/public/qiuqizhi/LOREM/datasets/cumulene_test.xyz


In [7]:
def eval_checkpoint(ckpt_dir: Path, add_offset: bool = True):
    calc = Calculator.from_checkpoint(ckpt_dir, add_offset=add_offset)
    systems = read(TEST_XYZ, index=':')

    e_ref = np.array([s.get_potential_energy() / len(s) for s in systems], dtype=float)
    f_ref = np.concatenate([s.get_forces() for s in systems], axis=0)

    for s in systems:
        s.calc = calc

    e_pred = np.array([s.get_potential_energy() / len(s) for s in systems], dtype=float)
    f_pred = np.concatenate([s.get_forces() for s in systems], axis=0)

    return {
        'energy_rmse_meV_per_atom': float(np.sqrt(mean_squared_error(e_ref, e_pred)) * 1000.0),
        'energy_mae_meV_per_atom': float(mean_absolute_error(e_ref, e_pred) * 1000.0),
        'force_rmse_meV_per_A': float(np.sqrt(mean_squared_error(f_ref, f_pred)) * 1000.0),
        'force_mae_meV_per_A': float(mean_absolute_error(f_ref, f_pred) * 1000.0),
    }

# 注意：add_offset=True 与项目里 metrics_cumulene.py 一致
sog_eval = eval_checkpoint(SOG_CKPT, add_offset=True)
cu_eval = eval_checkpoint(CU_CKPT, add_offset=True)

eval_df = pd.DataFrame([
    {'model': 'LOREM-SOG', **sog_eval},
    {'model': 'LOREM-CU', **cu_eval},
])

eval_df

,model,energy_rmse_meV_per_atom,energy_mae_meV_per_atom,force_rmse_meV_per_A,force_mae_meV_per_A
0,LOREM-SOG,2.965086,0.543064,48.015991,13.771518
1,LOREM-CU,3.032654,0.727683,44.320537,11.028973


In [8]:
eval_cmp = pd.DataFrame({
    'metric': [
        'energy_mae_meV_per_atom',
        'energy_rmse_meV_per_atom',
        'force_mae_meV_per_A',
        'force_rmse_meV_per_A',
    ],
    'LOREM-SOG': [
        sog_eval['energy_mae_meV_per_atom'],
        sog_eval['energy_rmse_meV_per_atom'],
        sog_eval['force_mae_meV_per_A'],
        sog_eval['force_rmse_meV_per_A'],
    ],
    'LOREM-CU': [
        cu_eval['energy_mae_meV_per_atom'],
        cu_eval['energy_rmse_meV_per_atom'],
        cu_eval['force_mae_meV_per_A'],
        cu_eval['force_rmse_meV_per_A'],
    ],
})

eval_cmp['better'] = eval_cmp.apply(lambda r: 'LOREM-SOG' if r['LOREM-SOG'] < r['LOREM-CU'] else 'LOREM-CU', axis=1)
eval_cmp['delta(SOG-CU)'] = eval_cmp['LOREM-SOG'] - eval_cmp['LOREM-CU']

print('Eval-only 对比结果（重新前向评估）:')
for _, r in eval_cmp.iterrows():
    sign = '更低(更好)' if r['delta(SOG-CU)'] < 0 else '更高(更差)'
    print(f"- {r['metric']}: SOG={r['LOREM-SOG']:.6f}, CU={r['LOREM-CU']:.6f} -> {r['better']} ({sign})")

eval_cmp

Eval-only 对比结果（重新前向评估）:
- energy_mae_meV_per_atom: SOG=0.543064, CU=0.727683 -> LOREM-SOG (更低(更好))
- energy_rmse_meV_per_atom: SOG=2.965086, CU=3.032654 -> LOREM-SOG (更低(更好))
- force_mae_meV_per_A: SOG=13.771518, CU=11.028973 -> LOREM-CU (更高(更差))
- force_rmse_meV_per_A: SOG=48.015991, CU=44.320537 -> LOREM-CU (更高(更差))


,metric,LOREM-SOG,LOREM-CU,better,delta(SOG-CU)
0,energy_mae_meV_per_atom,0.543064,0.727683,LOREM-SOG,-0.184619
1,energy_rmse_meV_per_atom,2.965086,3.032654,LOREM-SOG,-0.067568
2,force_mae_meV_per_A,13.771518,11.028973,LOREM-CU,2.742546
3,force_rmse_meV_per_A,48.015991,44.320537,LOREM-CU,3.695454


## 六模型 eval-only 对比（mp2 + mp1）

在相同数据集上比较以下 4 个模型的 best checkpoint（`run/checkpoints/MAE_F`）：
- `lorem-sog-cu30-lr-mp2`
- `lorem-cu30-lr-mp2`
- `lorem-sog-cu30-lr-mp1-run2`
- `lorem-cu30-lr-mp1`

In [6]:
MODEL_DIRS = {
    'SOG-mp2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2'),
    'CU-mp2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp2'),
    'SOG-mp1-run2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-run2'),
    'CU-mp1': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp1'),
    'SOG-mp2-ldependent': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-ldependent'),
    'SOG-mp1-ldependent': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-ldependent'),
}

CKPTS = {k: v / 'run/checkpoints/MAE_F' for k, v in MODEL_DIRS.items()}

# 统一检查 checkpoint 文件
for name, ckpt in CKPTS.items():
    for p in [
        ckpt / 'model/model.msgpack',
        ckpt / 'model/model.yaml',
        ckpt / 'model/baseline.yaml',
    ]:
        if not p.exists():
            raise FileNotFoundError(f'[{name}] Missing: {p}')

print('六模型 checkpoint 已就绪:')
for name, ckpt in CKPTS.items():
    print(f'- {name}: {ckpt}')

六模型 checkpoint 已就绪:
- SOG-mp2: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2/run/checkpoints/MAE_F
- CU-mp2: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp2/run/checkpoints/MAE_F
- SOG-mp1-run2: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-run2/run/checkpoints/MAE_F
- CU-mp1: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp1/run/checkpoints/MAE_F
- SOG-mp2-ldependent: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-ldependent/run/checkpoints/MAE_F
- SOG-mp1-ldependent: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-ldependent/run/checkpoints/MAE_F


In [7]:
def eval_suite(model_ckpts: dict, xyz_path: Path, add_offset: bool = True):
    rows = []
    for name, ckpt in model_ckpts.items():
        m = eval_checkpoint_on_xyz(ckpt, xyz_path, add_offset=add_offset)
        rows.append({'model': name, **m})
    return pd.DataFrame(rows)

# 兜底：如果未先运行上游单元，这里自动补齐数据路径
if 'TEST_XYZ' not in globals():
    TEST_XYZ = Path('/data/home/public/qiuqizhi/LOREM/datasets/cumulene_test.xyz')
if 'PROFILE_XYZ' not in globals():
    PROFILE_XYZ = Path('/data/home/public/qiuqizhi/LOREM/datasets/cumulene_profile.xyz')
for p in [TEST_XYZ, PROFILE_XYZ]:
    if not p.exists():
        raise FileNotFoundError(f'Missing dataset: {p}')

# 1) 在 cumulene_test.xyz 上六模型对比
four_test_df = eval_suite(CKPTS, TEST_XYZ, add_offset=True)
print('=== Six-model eval on cumulene_test.xyz ===')
display(four_test_df.sort_values('energy_rmse_meV_per_atom'))

# 2) 在 cumulene_profile.xyz（角度文件）上六模型对比
four_profile_df = eval_suite(CKPTS, PROFILE_XYZ, add_offset=True)
print('=== Six-model eval on cumulene_profile.xyz ===')
display(four_profile_df.sort_values('energy_rmse_meV_per_atom'))

# 附：按指标各自取最优
metrics_cols = [
    'energy_mae_meV_per_atom',
    'energy_rmse_meV_per_atom',
    'force_mae_meV_per_A',
    'force_rmse_meV_per_A',
]

best_test = {m: four_test_df.loc[four_test_df[m].idxmin(), ['model', m]].to_dict() for m in metrics_cols}
best_profile = {m: four_profile_df.loc[four_profile_df[m].idxmin(), ['model', m]].to_dict() for m in metrics_cols}

print('--- Best on cumulene_test.xyz ---')
for m, v in best_test.items():
    print(f"{m}: {v['model']} ({v[m]:.6f})")

print('--- Best on cumulene_profile.xyz ---')
for m, v in best_profile.items():
    print(f"{m}: {v['model']} ({v[m]:.6f})")

NameError: name 'eval_checkpoint_on_xyz' is not defined

## 六模型统一对比（四模型 checkpoint + 原文 CU 两模型）

这里将以下结果统一到一张表：
- 四模型 `four_profile_df`（来自 `cumulene_profile.xyz` eval-only）
- 原文 `CU-mp1/CU-mp2`（来自 `experiments/.../cumulene_dihedral_energy_curve.npz`）

> 说明：原文 `npz` 只有能量曲线，没有力标签，因此其 force 指标记为 `NaN`。

In [14]:
# 依赖：
# - four_profile_df: 四模型在 cumulene_profile.xyz 上的评估结果（上文已计算）
# - metrics_from_curve_npz: 原文 npz 的能量指标计算函数（上文已定义）

paper_rows = []
for name, d in PAPER_CU_DIRS.items():
    npz = d / 'cumulene_dihedral_energy_curve.npz'
    m = metrics_from_curve_npz(npz)
    paper_rows.append({
        'model': name,
        'energy_rmse_meV_per_atom': m['energy_rmse_meV_per_atom'],
        'energy_mae_meV_per_atom': m['energy_mae_meV_per_atom'],
        'force_rmse_meV_per_A': np.nan,
        'force_mae_meV_per_A': np.nan,
        'source': 'paper npz',
    })

paper_df = pd.DataFrame(paper_rows)

# 四模型结果补充 source，便于区分来源
four_profile_df_with_src = four_profile_df.copy()
four_profile_df_with_src['source'] = 'myexp checkpoint (profile xyz)'

# 六模型总表
six_models_df = pd.concat([
    four_profile_df_with_src[[
        'model',
        'energy_rmse_meV_per_atom',
        'energy_mae_meV_per_atom',
        'force_rmse_meV_per_A',
        'force_mae_meV_per_A',
        'source',
    ]],
    paper_df,
], ignore_index=True)

print('=== 六模型对比（统一按能量指标可比）===')
display(six_models_df.sort_values('energy_rmse_meV_per_atom'))

print('--- 按 energy_rmse_meV_per_atom 排名 ---')
for i, (_, r) in enumerate(six_models_df.sort_values('energy_rmse_meV_per_atom').iterrows(), 1):
    print(f"{i:>2}. {r['model']}: {r['energy_rmse_meV_per_atom']:.6f} meV/atom [{r['source']}]")

print('--- 按 energy_mae_meV_per_atom 排名 ---')
for i, (_, r) in enumerate(six_models_df.sort_values('energy_mae_meV_per_atom').iterrows(), 1):
    print(f"{i:>2}. {r['model']}: {r['energy_mae_meV_per_atom']:.6f} meV/atom [{r['source']}]")

=== 六模型对比（统一按能量指标可比）===


,model,energy_rmse_meV_per_atom,energy_mae_meV_per_atom,force_rmse_meV_per_A,force_mae_meV_per_A,source
0,SOG-mp2,0.271107,0.174035,13.150530,5.358935,myexp checkpoint (profile xyz)
1,CU-mp2,0.765605,0.576540,34.782138,4.494892,myexp checkpoint (profile xyz)
3,CU-mp1,1.501706,1.316303,36.496322,16.294766,myexp checkpoint (profile xyz)
2,SOG-mp1-run2,2.649827,2.567132,32.073229,13.456330,myexp checkpoint (profile xyz)
4,CU-mp2 (paper npz),8.922199,6.702351,NaN,NaN,paper npz
5,CU-mp1 (paper npz),10.981008,9.210839,NaN,NaN,paper npz


--- 按 energy_rmse_meV_per_atom 排名 ---
 1. SOG-mp2: 0.271107 meV/atom [myexp checkpoint (profile xyz)]
 2. CU-mp2: 0.765605 meV/atom [myexp checkpoint (profile xyz)]
 3. CU-mp1: 1.501706 meV/atom [myexp checkpoint (profile xyz)]
 4. SOG-mp1-run2: 2.649827 meV/atom [myexp checkpoint (profile xyz)]
 5. CU-mp2 (paper npz): 8.922199 meV/atom [paper npz]
 6. CU-mp1 (paper npz): 10.981008 meV/atom [paper npz]
--- 按 energy_mae_meV_per_atom 排名 ---
 1. SOG-mp2: 0.174035 meV/atom [myexp checkpoint (profile xyz)]
 2. CU-mp2: 0.576540 meV/atom [myexp checkpoint (profile xyz)]
 3. CU-mp1: 1.316303 meV/atom [myexp checkpoint (profile xyz)]
 4. SOG-mp1-run2: 2.567132 meV/atom [myexp checkpoint (profile xyz)]
 5. CU-mp2 (paper npz): 6.702351 meV/atom [paper npz]
 6. CU-mp1 (paper npz): 9.210839 meV/atom [paper npz]
